# Figure: Shorkie LM TF-MoDISco motif logos

This figure shows the sequence-motif vocabulary the masked DNA language model (`Shorkie LM`) learns over fungal genomes. We run TF-MoDISco on the LM's per-base attribution scores and render the discovered patterns as logos: the **contribution-weighted matrix (CWM)** logo (signed importance per base) and the matched **per-position-IC PWM** logo (information-content-scaled letter heights). The top positive patterns recover canonical yeast regulatory motifs (e.g. poly(A)/AT-rich, TATA-like, and GC-rich elements).

**Reproduces:** TF-MoDISco motif logos for the Shorkie LM (CWM + IC-weighted PWM for the top patterns).

**Upstream:** `scripts/04_analysis/shorkie_lm/motif_analysis/motif_lm/1_search_motif.py` then `2_modisco_script.sh` run `modisco motifs` on the LM attribution tensors and write `modisco_results_w16384_n100000.h5` under `results.modisco_lm/<model_arch>/`. This notebook loads that `.h5`; it does NOT regenerate it.

**Requires:** `yeast_ml` conda env + `pip install -e .`; plus `logomaker` (`pip install logomaker`). No GPU needed (reads a precomputed `.h5`).

**Source script:** ported from `scripts/04_analysis/shorkie_lm/motif_analysis/motif_lm/4_viz_motif.py` (CWM/PWM logo rendering); the seqlet-overlay variant in `5_modisco_DNA_logo/1_modisco_DNA_logo.py` is not reproduced here.

In [ ]:
# Imports: stdlib / 3rd-party
import os
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
import logomaker  # pip install logomaker

# shorkie public API
from shorkie import config
from shorkie.helpers.yeast_helpers import dna_letter_at  # optional fallback logo renderer

%matplotlib inline

In [ ]:
# Path resolution via shorkie.config (no hardcoded paths)
config.load()

# Which LM model architecture's modisco results to visualize.
# The modisco stage writes one .h5 per arch under results.modisco_lm/<model_arch>/.
MODEL_ARCH = "unet_small_bert_drop"
MODISCO_FNAME = "modisco_results_w16384_n100000.h5"

modisco_root = config.path("results.modisco_lm")
modisco_h5 = os.path.join(str(modisco_root), MODEL_ARCH, MODISCO_FNAME)
print("motif_db_dir :", config.path("motif_db_dir"))
print("modisco_h5   :", modisco_h5)
print("exists       :", os.path.isfile(modisco_h5))

In [ ]:
# Helper: per-position information content (bits), ported from 4_viz_motif.py
def compute_per_position_ic(ppm, background, pseudocount):
    alphabet_len = len(background)
    ic = (
        (np.log((ppm + pseudocount) / (1 + pseudocount * alphabet_len)) / np.log(2)) * ppm
        - (np.log(background) * background / np.log(2))[None, :]
    )
    return np.sum(ic, axis=1)


# Helper: trim a pattern to its informative core (>= trim_threshold of peak abs-contrib),
# with a small flank. Mirrors the trimming logic in 4_viz_motif.py.
def trim_pattern(cwm, trim_threshold=0.3, flank=4):
    score = np.sum(np.abs(cwm), axis=1)
    thresh = np.max(score) * trim_threshold
    pass_inds = np.where(score >= thresh)[0]
    if len(pass_inds) == 0:
        return 0, cwm.shape[0]
    start = max(np.min(pass_inds) - flank, 0)
    end = min(np.max(pass_inds) + flank + 1, cwm.shape[0])
    return start, end

In [ ]:
# Load the TF-MoDISco patterns. For each pattern we keep its CWM (contrib_scores)
# and PWM (sequence) plus the seqlet count, sorted by pattern index.
BACKGROUND = np.array([0.25, 0.25, 0.25, 0.25])
PATTERN_GROUPS = ["pos_patterns", "neg_patterns"]

patterns = []
with h5py.File(modisco_h5, "r") as f:
    for grp in PATTERN_GROUPS:
        if grp not in f:
            continue
        metacluster = f[grp]
        # sort by trailing integer in pattern_<k>
        for pname in sorted(metacluster.keys(), key=lambda x: int(x.split("_")[-1])):
            pat = metacluster[pname]
            cwm = np.array(pat["contrib_scores"][:])      # (L, 4) signed importance
            ppm = np.array(pat["sequence"][:])            # (L, 4) probabilities
            n_seqlets = int(pat["seqlets"]["n_seqlets"][0]) if "seqlets" in pat else None
            patterns.append({
                "tag": f"{grp}.{pname}",
                "group": grp,
                "cwm": cwm,
                "ppm": ppm,
                "n_seqlets": n_seqlets,
            })

print(f"Loaded {len(patterns)} TF-MoDISco patterns from {os.path.basename(modisco_h5)}")
for p in patterns[:6]:
    print(f"  {p['tag']:24s} L={p['cwm'].shape[0]:3d}  n_seqlets={p['n_seqlets']}")

In [ ]:
# Render the top positive patterns as paired logos: CWM (signed contribution)
# and IC-weighted PWM. Ported from _plot_weights/make_logo in 4_viz_motif.py,
# using logomaker for clean letter stacking.
LETTER_COLS = ["A", "C", "G", "T"]
TOP_N = 4

pos_patterns = [p for p in patterns if p["group"] == "pos_patterns"][:TOP_N]

fig, axes = plt.subplots(len(pos_patterns), 2, figsize=(14, 2.4 * len(pos_patterns)))
if len(pos_patterns) == 1:
    axes = axes[None, :]

for r, p in enumerate(pos_patterns):
    cwm = p["cwm"]
    ppm = p["ppm"]
    s, e = trim_pattern(cwm)

    # Left: CWM logo (signed contribution scores)
    df_cwm = pd.DataFrame(cwm[s:e], columns=LETTER_COLS)
    df_cwm.index.name = "pos"
    lg = logomaker.Logo(df_cwm, ax=axes[r, 0])
    lg.style_spines(visible=False)
    axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])
    axes[r, 0].set_ylabel(p["tag"], fontsize=9)
    if r == 0:
        axes[r, 0].set_title("CWM (contribution)")

    # Right: IC-weighted PWM logo
    ic = compute_per_position_ic(ppm, BACKGROUND, pseudocount=0.001)
    df_pwm = pd.DataFrame(ppm[s:e] * ic[s:e, None], columns=LETTER_COLS)
    df_pwm.index.name = "pos"
    lg2 = logomaker.Logo(df_pwm, ax=axes[r, 1])
    lg2.style_spines(visible=False)
    axes[r, 1].set_xticks([]); axes[r, 1].set_yticks([])
    if r == 0:
        axes[r, 1].set_title("PWM (information content)")

fig.suptitle(f"Shorkie LM TF-MoDISco motifs ({MODEL_ARCH})", y=1.0, fontsize=12)
plt.tight_layout()
plt.show()

## Notes

- **Two logo views per motif.** The left column is the **CWM** (signed `contrib_scores`): letter height reflects how much each base drives the LM's attribution at that position, including negative (penalizing) bases. The right column is the **PWM** (`sequence` probabilities) scaled by per-position information content, the conventional bits logo.
- **Trimming.** Each pattern is trimmed to its informative core (positions with abs-contribution >= 30% of the per-pattern peak, padded by 4bp), matching the `trim_threshold`/flank logic in `4_viz_motif.py`.
- **Choosing the model arch.** The LM modisco run produces one `.h5` per architecture/replicate (e.g. `unet_small_bert_drop`, `..._retry_1`, `..._retry_2`). Set `MODEL_ARCH` to the one you want; the released figure uses `unet_small_bert_drop`.
- **To regenerate the input artifact** (not runnable from released data alone): run the upstream attribution + modisco stage (`1_search_motif.py` then `2_modisco_script.sh`) to write `modisco_results_w16384_n100000.h5` under `results.modisco_lm/<model_arch>/`.
- `dna_letter_at` is imported from `shorkie.helpers.yeast_helpers` as a fallback renderer if `logomaker` is unavailable; the primary path uses `logomaker` to faithfully match the source script.